# Cleaning de BD Consolidada

In [1]:
import pandas as pd

In [13]:
data = pd.read_parquet("/Users/adrianmateos/Documents/RetoOctavo/data_consolidation/output/transacciones_consolidado.parquet")

print("shape datos consolidados:", data.shape)

shape datos consolidados: (1145526, 57)


In [3]:
print(data.columns)

Index(['t_folio', 't_folio_ext', 't_referencia', 't_transaccion', 't_codigo',
       't_cve_res', 't_cuarto', 't_centro_consumo', 't_fecha', 't_tra_hra',
       't_tra_mto', 't_monto', 't_impuesto', 't_impuesto2', 't_propina',
       't_carabo', 't_tra_cancelada', 't_can_dia', 't_can_mes', 't_can_hra',
       't_can_mto', 't_usuario', 't_usuario_mod', 't_num_adu', 't_num_per',
       't_noches', 't_inc_tfa', 't_observaciones', 't_timestamp', 'es_split',
       'es_renta', 'h_status', 'h_tpo_hab', 'h_tpo_hsp', 'h_seg_mer',
       'h_cod_age', 'h_tpo_plan', 'h_for_pgo', 'h_tpo_mon', 'h_num_per',
       'h_num_adu', 'h_num_men', 'h_num_noc', 'h_tfa', 'h_tfa_total',
       'h_tfa_renta', 'h_tfa_impuestos', 'h_tfa_extras', 'h_tarifa_forzada',
       'h_dep_sol', 'h_lim_cre', 'h_fec_lld', 'h_fec_sda', 'h_fec_reg',
       'h_res_usr', 'h_rec_usr', 'tiene_reservacion'],
      dtype='str')


In [4]:
# Número de transacciones por día
transacciones_por_dia = data.groupby(data["t_fecha"]).size()
print("Promedio de transacciones por día:", transacciones_por_dia.mean())

# Número de transacciones por mes
transacciones_por_mes = data.groupby(data["t_fecha"].dt.to_period("M")).size()
print("Promedio de transacciones por mes:", transacciones_por_mes.mean())

# Número de transacciones por año
transacciones_por_año = data.groupby(data["t_fecha"].dt.year).size()
print("Promedio de transacciones por año:", transacciones_por_año.mean())

Promedio de transacciones por día: 622.9070146818923
Promedio de transacciones por mes: 18476.225806451614
Promedio de transacciones por año: 190921.0


In [5]:
data["t_centro_consumo"].value_counts()

t_centro_consumo
00    865829
01       302
Name: count, dtype: int64[pyarrow]

In [6]:
print("% de transacciones que ocurren en recepcion (codigo 00):", round(865829/(865829 + 302), 4))

% de transacciones que ocurren en recepcion (codigo 00): 0.9997


In [7]:
print("Valores unicos de codigo de concepto del cargo: ", data['t_codigo'].nunique())

# Calcula el porcentaje de cada valor único en la columna t_codigo
porcentajes_t_codigo = data['t_codigo'].value_counts(normalize=True) * 100
print("primeros 10:\n", porcentajes_t_codigo.head(10))

# Imprime los códigos que representan un porcentaje >= 2.5%
codigos_mayor_igual_25 = porcentajes_t_codigo[porcentajes_t_codigo >= 2.5]
print("Códigos con >= 2.5% de las transacciones:")
for codigo, porcentaje in codigos_mayor_igual_25.items():
    print(f"{codigo}: {porcentaje:.2f}%")

# Calcula y muestra el porcentaje total del resto de los códigos (< 2.5%)
resto_porcentaje = porcentajes_t_codigo[porcentajes_t_codigo < 2.5].sum()
print(f"Porcentaje de las transacciones que representan el resto de los códigos (<2.5% cada uno): {resto_porcentaje:.2f}%")

Valores unicos de codigo de concepto del cargo:  108


primeros 10:
 t_codigo
PROPTI    42.964367
RENHAB    30.039737
DEPHAB      5.17282
RESDEP       4.7799
TRANSF     2.637653
TARCRE     2.611726
EFE        2.159968
TARDEB     1.727853
RENCOM     0.963051
CANXFA     0.930664
Name: proportion, dtype: double[pyarrow]
Códigos con >= 2.5% de las transacciones:
PROPTI: 42.96%
RENHAB: 30.04%
DEPHAB: 5.17%
RESDEP: 4.78%
TRANSF: 2.64%
TARCRE: 2.61%
Porcentaje de las transacciones que representan el resto de los códigos (<2.5% cada uno): 11.79%


## Hallazgos adicionales y limpieza estructural

Conclusiones sobre los dos ejemplos iniciales y hallazgos nuevos, **todos verificados en las celdas de abajo**. La limpieza reproducible vive en `clean.py` (genera `output/transacciones_limpio.parquet`).

In [8]:
# Casi-constantes + F1: evidencia para eliminar columnas sin varianza / redundantes
import numpy as np
n = len(data)
prof = []
for c in data.columns:
    nn = data[c].dropna()
    top = (nn.value_counts(normalize=True).iloc[0] * 100) if len(nn) else 0
    prof.append((c, round(100*(1 - len(nn)/n), 1), nn.nunique(), round(top, 2)))
prof = pd.DataFrame(prof, columns=["col", "null%", "num_uniq", "top%_nonnull"])
print("Casi-constantes (top%>=98 sobre no-nulos) o casi-vacias (null%>=95) -> candidatas a DROP:")
print(prof[(prof["top%_nonnull"] >= 98) | (prof["null%"] >= 95)].to_string(index=False))

# F1: centro '01' es EXACTAMENTE el codigo PUNTAZ -> columna redundante con t_codigo
print("\nF1 | centro '01':", (data['t_centro_consumo'] == '01').sum(),
      "| de ellos PUNTAZ:", ((data['t_centro_consumo'] == '01') & (data['t_codigo'] == 'PUNTAZ')).sum(),
      "| PUNTAZ totales:", (data['t_codigo'] == 'PUNTAZ').sum())

Casi-constantes (top%>=98 sobre no-nulos) o casi-vacias (null%>=95) -> candidatas a DROP:
             col  null%  num_uniq  top%_nonnull
t_centro_consumo   24.4         2         99.97
        es_split    0.0         2         99.99
        h_status   21.5         2         99.98
       h_tpo_mon   21.5         2         99.99
       h_lim_cre   21.5         5         99.74

F1 | centro '01': 302 | de ellos PUNTAZ: 302 | PUNTAZ totales: 302


In [9]:
# F2-F4: redundancia / colinealidad entre columnas numericas (|r|>=0.9) -> conservar solo una
num = ["t_monto", "t_impuesto", "t_impuesto2", "t_propina", "t_num_per", "t_num_adu",
       "h_num_per", "h_num_adu", "h_tfa", "h_tfa_total", "h_tfa_renta", "h_tfa_impuestos"]
corr = data[num].astype("float64").corr()
print("Pares casi redundantes (|r|>=0.9):")
for i in range(len(num)):
    for j in range(i + 1, len(num)):
        if abs(corr.iloc[i, j]) >= 0.9:
            print(f"  {num[i]:16s} ~ {num[j]:16s} r={corr.iloc[i, j]:.3f}")

# h_tfa_impuestos NO es impuestos: es un duplicado mal nombrado de h_tfa_total
sub = data.dropna(subset=["h_tfa_total", "h_tfa_impuestos"])
print(f"\nh_tfa_impuestos == h_tfa_total en {100*(abs(sub['h_tfa_total']-sub['h_tfa_impuestos'])<=1).mean():.1f}%"
      " -> duplicado mal nombrado")

Pares casi redundantes (|r|>=0.9):
  t_impuesto       ~ t_impuesto2      r=0.920
  t_num_per        ~ h_num_per        r=0.997
  t_num_adu        ~ h_num_adu        r=0.991
  h_tfa_total      ~ h_tfa_renta      r=0.992
  h_tfa_total      ~ h_tfa_impuestos  r=0.983
  h_tfa_renta      ~ h_tfa_impuestos  r=0.986

h_tfa_impuestos == h_tfa_total en 97.3% -> duplicado mal nombrado


In [10]:
# F6: el "impuesto ~16%" NO aplica en general (el impuesto vive casi solo en renta)
m = data[data["t_monto"] > 0]
print(f"F6 | % cargos (monto>0) con impuesto=0: {100*(m['t_impuesto']==0).mean():.1f}%"
      f" | mediana impuesto/monto: {(m['t_impuesto']/m['t_monto']).median():.3f}")

# F7: t_carabo (0=cargo, 1=abono) NO equivale al signo del monto -> ambos son senal
print("\nF7 | t_carabo vs signo de t_monto:")
print(pd.crosstab(data["t_carabo"], np.sign(data["t_monto"])))

# F9: montos cero y negativos = senal, no basura -> NO filtrar
print(f"\nF9 | monto==0: {(data['t_monto']==0).sum():,} | monto<0: {(data['t_monto']<0).sum():,} -> se conservan")

F6 | % cargos (monto>0) con impuesto=0: 57.1% | mediana impuesto/monto: 0.000

F7 | t_carabo vs signo de t_monto:
t_monto    -1.0    0.0     1.0
t_carabo                      
0         53979  14266  881409
1          5954   1154  188764

F9 | monto==0: 15,420 | monto<0: 59,933 -> se conservan


In [11]:
# F10: NO hay duplicados 100% reales (t_transaccion es ID unico secuencial).
# El "cargo duplicado" del user story es un FEATURE derivado, no un dedup.
k = ["t_folio", "t_folio_ext", "t_codigo", "t_monto", "t_fecha"]
print(f"Comparten (folio,sub,codigo,monto,fecha):   {100*data.duplicated(k, keep=False).mean():.1f}%")
print(f"... + mismo timestamp (minuto):             {100*data.duplicated(k+['t_timestamp'], keep=False).mean():.1f}%")
print(f"... + t_transaccion (identicas al 100%):    {data.duplicated(k+['t_timestamp','t_transaccion'], keep=False).sum()} filas")

Comparten (folio,sub,codigo,monto,fecha):   27.5%
... + mismo timestamp (minuto):             26.5%
... + t_transaccion (identicas al 100%):    0 filas


### Pase 1 — limpieza estructural (57 → 47 columnas)

**10 columnas eliminadas** (redundantes / constantes / colineales) y **0 filas** (en no-supervisado los casos raros se conservan):

| Eliminada | Motivo |
|---|---|
| `t_centro_consumo` | ⟺ código PUNTAZ; 24% nula |
| `h_tpo_mon` | constante (MXN) |
| `h_lim_cre` | 99.7% en 0 + centinelas |
| `h_tfa_impuestos` | duplicado de `h_tfa_total` |
| `h_tfa_renta` | colineal r=0.99 |
| `t_impuesto2` | colineal r=0.92 |
| `t_num_per`, `t_num_adu` | duplican `h_num_*` (más nulos / menos cobertura) |
| `t_tra_hra`, `t_tra_mto` | ya están dentro de `t_timestamp` |

Reproducible con `uv run python data_cleaning/clean.py` (dict `DROP_REDUNDANTE`). Sigue el **Pase 2** abajo.

## Pase 2 — ¿de verdad aportan las 47 columnas? (filtro por valor para anomalías)

Segundo filtro, ya pensando en el modelo. Para cada columna nos preguntamos: *¿aporta a detectar una anomalía financiera, ya sea como feature del Isolation Forest, como regla, o como trazabilidad para el auditor?* La respuesta honesta: **13 no aportan**. Evidencia abajo.

In [12]:
# Evidencia del Pase 2 (sobre la base consolidada `data`)
print("h_status (solo salida/en casa -> SIN cancelacion/no-show, inutil):")
print(data["h_status"].value_counts(dropna=False).to_string())

occ = data.dropna(subset=["h_num_per", "h_num_adu", "h_num_men"])
print(f"\nOcupacion redundante: h_num_per == adu+men en "
      f"{100*((occ['h_num_per']-(occ['h_num_adu']+occ['h_num_men'])).abs()<=0).mean():.1f}%")

print(f"t_fecha redundante: normalize(t_timestamp) == t_fecha en "
      f"{100*(data['t_timestamp'].dt.normalize()==data['t_fecha']).mean():.0f}%")

print("\nCasi-constantes (% del valor dominante) -> inutiles para el IF:")
for c in ["t_inc_tfa", "h_tfa_extras"]:
    print(f"  {c:14s} {data[c].dropna().value_counts(normalize=True).iloc[0]*100:.1f}%")

print("\nCancelacion troceada (sin año) -> ya resumida por el flag t_tra_cancelada:")
for c in ["t_can_dia", "t_can_mes", "t_can_hra", "t_can_mto"]:
    print(f"  {c:12s} {100*data[c].isna().mean():.1f}% nulo")

h_status (solo salida/en casa -> SIN cancelacion/no-show, inutil):
h_status
50      899461
<NA>    245858
10         207

Ocupacion redundante: h_num_per == adu+men en 99.9%
t_fecha redundante: normalize(t_timestamp) == t_fecha en 100%

Casi-constantes (% del valor dominante) -> inutiles para el IF:
  t_inc_tfa      97.5%
  h_tfa_extras   97.4%

Cancelacion troceada (sin año) -> ya resumida por el flag t_tra_cancelada:
  t_can_dia    85.4% nulo
  t_can_mes    85.4% nulo
  t_can_hra    85.4% nulo
  t_can_mto    85.4% nulo


### Conclusión del Pase 2 → 47 − 13 = **34 columnas**

No todas las 47 aportaban. Se eliminan **13 más**:

| Eliminada | Motivo |
|---|---|
| `t_fecha` | redundante con `t_timestamp` (100%) |
| `t_inc_tfa` | casi-constante (97.5% 'S') + 26% nula |
| `h_status` | solo salida/en casa; **sin** señal de cancelación/no-show |
| `h_tfa_extras` | casi-constante (97.4% en 0) |
| `h_num_adu`, `h_num_men` | `h_num_per = adu + men` (99.9%) → redundantes |
| `h_res_usr`, `h_rec_usr` | usuarios de la reserva; débiles para el cargo |
| `h_fec_reg` | fecha de registro de reserva; poca relevancia al cargo |
| `t_can_dia/mes/hra/mto` | 85% nulas, sin año; cancelación ya en `t_tra_cancelada` |

**Se conservan `es_split` y `t_tra_cancelada`** aunque estén desbalanceadas: marcan eventos con sentido de *regla* (doble conteo / cargo cancelado), no son ruido como `h_status`. Los **IDs** y `t_observaciones` se quedan para derivar features y trazabilidad del auditor (no entran como feature crudo).

Base final: **34 columnas**, 1,145,526 filas → `output/transacciones_limpio.parquet` (dicts `DROP_REDUNDANTE` + `DROP_BAJO_VALOR` en `clean.py`).

**Pendiente (fase de modelado, NO limpieza):** imputar nulos (IF no acepta NaN), encoding por **frecuencia**, **escalar `t_monto` por `t_codigo`** (mayor impacto) y feature `n_duplicados`.